# Task 1: BestBuy Canada Product Review Scraping

## Objective
Collect product reviews from BestBuy Canada, structure the review data, perform sentiment analysis, extract useful tags, and summarize business insights.

## Deliverables
- Structured review dataset
- Review sentiment analysis and tags
- Scraping challenges and limitations
- Business insights and recommendations

In [1]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
from datetime import datetime
from pathlib import Path
from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import requests

url = "https://www.bestbuy.ca/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("HTTP status:", response.status_code)
print("Page received:", len(response.text), "characters")

HTTP status: 200
Page received: 464830 characters


In [3]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

soup = BeautifulSoup(response.text, "html.parser")

product_links = []
for a in soup.select("a[href]"):
    href = a.get("href", "")
    text = a.get_text(" ", strip=True)

    if "/en-ca/product/" in href and text:
        product_links.append({
            "product_name": text,
            "product_url": urljoin("https://www.bestbuy.ca", href)
        })

products_df = pd.DataFrame(product_links).drop_duplicates(
    subset="product_url"
)

print("Product links found:", len(products_df))
display(products_df.head(10))

Product links found: 0


""


In [4]:
deals_url = "https://www.bestbuy.ca/en-ca/collection/shop-all-deals/16074"

deals_response = requests.get(
    deals_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("HTTP status:", deals_response.status_code)
print("Page size:", len(deals_response.text))

deals_soup = BeautifulSoup(deals_response.text, "html.parser")

links = []
for a in deals_soup.select("a[href]"):
    href = a.get("href", "")
    name = a.get_text(" ", strip=True)

    if name and "/en-ca/" in href and "/seller/" not in href:
        links.append({
            "name": name,
            "url": urljoin("https://www.bestbuy.ca", href)
        })

deals_df = pd.DataFrame(links).drop_duplicates(subset="url")
display(deals_df.head(15))

HTTP status: 403
Page size: 427


""


In [5]:
product_url = (
    "https://www.bestbuy.ca/en-ca/product/"
    "google-pixel-9a-128gb-iris-unlocked/18933645"
)

print("Selected product: Google Pixel 9a 128GB Iris")
print("Product URL:", product_url)

Selected product: Google Pixel 9a 128GB Iris
Product URL: https://www.bestbuy.ca/en-ca/product/google-pixel-9a-128gb-iris-unlocked/18933645


In [6]:
product_response = requests.get(
    product_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("HTTP status:", product_response.status_code)
print("Response size:", len(product_response.text), "characters")
print(
    "Review-related text present:",
    "review" in product_response.text.lower()
)

HTTP status: 403
Response size: 460 characters
Review-related text present: False


### Scraping Access Limitation

The selected product is Google Pixel 9a 128GB Iris Unlocked. Manual inspection of the Best Buy Canada product page showed 58 customer reviews. However, the direct Python HTTP request returned status code 403, and the response did not contain review-related text.

This indicates that the website is restricting the automated request. No reviews will be fabricated or reported as successfully scraped. Further collection will use only publicly accessible content and permitted methods.

first_review = {
    "product_title": "Google Pixel 9a 128GB Iris - Unlocked",
    "review_title": "We all love Google phone now",
    "reviewer": "jennychauho_9206",
    "review_date": "2025-05-19",
    "rating": 5,
    "source": "Best Buy Canada",
    "review_text": (
        "I gave for my mom this phone, she love it "
        "(but we thought it will be purple color, but they sent us black). "
        "It really easy to set up and get all things from old phone to new "
        "phone. Took 30 mins from we open the box to finished set up "
        "(ready to use). For me it's really fast. And they have good camera."
    )
}

print("Review captured from the visible product page:")
display(pd.DataFrame([first_review]))

In [7]:
first_review = {
    "product_title": "Google Pixel 9a 128GB Iris - Unlocked",
    "review_title": "We all love Google phone now",
    "reviewer": "jennychauho_9206",
    "review_date": "2025-05-19",
    "rating": 5,
    "source": "Best Buy Canada",
    "collection_method": "manual transcription from screenshot",
    "review_text": (
        "I gave for my mom this phone, she love it "
        "(but we thought it will be purple color, but they sent us black). "
        "It really easy to set up and get all things from old phone to new phone. "
        "Took 30 mins from we open the box to finished set up (ready to use). "
        "For me it's really fast. And they have good camera."
    )
}

print("Review record prepared.")
display(pd.DataFrame([first_review]))

Review record prepared.


,product_title,review_title,reviewer,review_date,rating,source,collection_method,review_text
0,Google Pixel 9a 128GB Iris - Unlocked,We all love Google phone now,jennychauho_9206,2025-05-19,5,Best Buy Canada,manual transcription from screenshot,"I gave for my mom this phone, she love it (but..."


In [8]:
# Create a DataFrame from the review record
reviews_df = pd.DataFrame([first_review])

# Save the CSV inside the Task_1_Scraping folder
output_dir = Path("..") / "Final_Submission"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "HanuAi_Task1_Reviews.csv"
reviews_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV saved:", csv_path.resolve())
print("Number of review records:", len(reviews_df))
print("Number of columns:", len(reviews_df.columns))

CSV saved: C:\Users\manoj\OneDrive\Desktop\HanuAi_ML_Assessment\Final_Submission\HanuAi_Task1_Reviews.csv
Number of review records: 1
Number of columns: 8


In [9]:
# Basic sentiment analysis and keyword tags
def analyze_sentiment(text):
    text = str(text).lower()

    positive_words = [
        "love", "good", "great", "easy", "fast", "excellent", "happy"
    ]
    negative_words = [
        "bad", "poor", "slow", "broken", "problem", "issue", "wrong"
    ]

    positive_score = sum(word in text for word in positive_words)
    negative_score = sum(word in text for word in negative_words)

    if positive_score > negative_score:
        return "Positive"
    elif negative_score > positive_score:
        return "Negative"
    return "Neutral"


def extract_tags(text):
    text = str(text).lower()
    tag_keywords = {
        "Setup": ["setup", "set up", "install"],
        "Performance": ["fast", "slow", "speed"],
        "Camera": ["camera", "photo", "picture"],
        "Delivery / Color": ["sent us", "color", "colour"],
        "Ease of Use": ["easy", "simple", "use"]
    }

    return [
        tag for tag, keywords in tag_keywords.items()
        if any(keyword in text for keyword in keywords)
    ]


reviews_df["sentiment"] = reviews_df["review_text"].apply(analyze_sentiment)
reviews_df["tags"] = reviews_df["review_text"].apply(extract_tags)

display(reviews_df[["review_title", "rating", "sentiment", "tags"]])

,review_title,rating,sentiment,tags
0,We all love Google phone now,5,Positive,"[Setup, Performance, Camera, Delivery / Color,..."


In [10]:
# Save the enriched review dataset
reviews_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Also save an Excel version
excel_path = output_dir / "HanuAi_Task1_Reviews.xlsx"
reviews_df.to_excel(excel_path, index=False)

print("Enriched CSV saved:", csv_path.name)
print("Excel file saved:", excel_path.name)
print("Rows:", len(reviews_df))
print("Columns:", len(reviews_df.columns))

Enriched CSV saved: HanuAi_Task1_Reviews.csv
Excel file saved: HanuAi_Task1_Reviews.xlsx
Rows: 1
Columns: 10


## Preliminary Findings and Limitations

### Product Overview
- **Product:** Google Pixel 9a 128GB Iris - Unlocked
- **Source:** Best Buy Canada
- **Displayed overall rating:** 4.6/5 from 58 reviews at the time of manual inspection
- **Review records in this dataset:** 1 manually transcribed review

### Preliminary Finding
The captured review has a 5-star rating and was classified as positive by a simple keyword-based sentiment rule. Its text mentions ease of setup, performance, camera quality, and a color mismatch. These findings apply only to this single example and cannot be generalized to all customers.

### Collection Limitations
The direct Python request to the product page returned HTTP 403. The notebook therefore does not claim that 58 reviews were scraped. The current dataset contains one manually transcribed example from the publicly visible page. Automated collection, pagination, and analysis across the full review set remain incomplete.

### Recommended Next Steps
Use a permitted data source or collection method to obtain a larger review sample. Then validate sentiment labels, analyze review trends, and draw business conclusions only from the collected records. Clearly document the source, collection method, and any access restrictions.